# Image Segmentation

## Overview

Segmentation assigns a label to every pixel in an image. It is more informative than classification (one label per image) and more granular than detection (one box per object).

**Three types:**

| Type | Output | Use case |
|---|---|---|
| Semantic | Class label per pixel | Land cover mapping |
| Instance | Class + unique ID per object | Counting individuals |
| Panoptic | Semantic + instance unified | Full scene understanding |

**U-Net architecture:** the dominant encoder-decoder architecture for semantic segmentation, especially in scientific imaging. Skip connections between encoder and decoder preserve fine-grained spatial detail lost during downsampling.

```
Input → Encoder (downsample) → Bottleneck → Decoder (upsample) → Segmentation map
             ↑_______skip connections_______↑
```

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy

torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simulate: 64x64 aerial patches with 3-class segmentation masks
# Classes: 0=water, 1=vegetation, 2=bare_soil
N_CLASSES = 3
IMG_SIZE  = 64

def make_seg_sample(seed=0, size=IMG_SIZE):
    r  = np.random.default_rng(seed)
    mask = np.zeros((size, size), dtype=np.int64)
    img  = np.zeros((size, size, 3), dtype=np.float32)
    # Water region (bottom)
    mask[size//2:, :] = 0
    img[size//2:, :]  = [0.2, 0.4, 0.7]
    # Vegetation (top-left)
    mask[:size//2, :size//2] = 1
    img[:size//2, :size//2]  = [0.2, 0.6, 0.2]
    # Bare soil (top-right)
    mask[:size//2, size//2:] = 2
    img[:size//2, size//2:]  = [0.6, 0.5, 0.3]
    # Add noise and slight irregularity
    img  += r.normal(0, 0.05, img.shape).astype(np.float32)
    img   = img.clip(0, 1)
    # Blur boundary slightly
    boundary_noise = r.normal(0, 3, (size, size)).astype(np.float32)
    return img, mask

class SegDataset(Dataset):
    def __init__(self, n_samples, offset=0):
        self.samples = [make_seg_sample(seed=i+offset) for i in range(n_samples)]
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img, mask = self.samples[i]
        return (torch.from_numpy(img.transpose(2,0,1)),   # CHW
                torch.from_numpy(mask))

train_ds = SegDataset(200, offset=0)
val_ds   = SegDataset(50,  offset=200)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=16)
Xb, yb   = next(iter(train_dl))
print(f"Batch: X={Xb.shape}, y={yb.shape}")
print(f"Classes: {yb.unique().tolist()}")

---
## U-Net Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU())
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=3, base_ch=32):
        super().__init__()
        b = base_ch
        # Encoder
        self.enc1 = DoubleConv(in_channels, b)
        self.enc2 = DoubleConv(b,   b*2)
        self.enc3 = DoubleConv(b*2, b*4)
        self.pool = nn.MaxPool2d(2)
        # Bottleneck
        self.bottleneck = DoubleConv(b*4, b*8)
        # Decoder
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*8, b*4)   # b*4 + b*4 skip
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*4, b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b)
        self.out  = nn.Conv2d(b, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(bn), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

unet  = UNet(in_channels=3, n_classes=N_CLASSES).to(device)
total = sum(p.numel() for p in unet.parameters())
print(f"U-Net parameters: {total:,}")
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(device)
out   = unet(dummy)
print(f"Input: {dummy.shape} -> Output: {out.shape}  (B, C, H, W)")
print("Output spatial dims match input -- pixel-wise classification")

---
## Training and Dice Loss

In [ ]:
def dice_loss(pred, target, n_classes=3, smooth=1.0):
    pred_soft = F.softmax(pred, dim=1)
    total = 0
    for cls in range(n_classes):
        p = pred_soft[:, cls]
        t = (target == cls).float()
        inter = (p * t).sum()
        total += 1 - (2*inter + smooth) / (p.sum() + t.sum() + smooth)
    return total / n_classes

def combo_loss(pred, target, alpha=0.5):
    return alpha * nn.CrossEntropyLoss()(pred, target) + (1-alpha) * dice_loss(pred, target)

optimiser = optim.AdamW(unet.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=40)
best_loss, best_state = float('inf'), None
tr_losses, va_losses  = [], []
for epoch in range(40):
    unet.train()
    ep_losses = []
    for Xb, yb in train_dl:
        Xb, yb = Xb.to(device), yb.to(device)
        optimiser.zero_grad()
        loss = combo_loss(unet(Xb), yb)
        loss.backward(); optimiser.step()
        ep_losses.append(loss.item())
    scheduler.step()
    unet.eval()
    with torch.no_grad():
        va_loss = np.mean([combo_loss(unet(Xb.to(device)), yb.to(device)).item()
                           for Xb, yb in val_dl])
    tr_losses.append(np.mean(ep_losses)); va_losses.append(va_loss)
    if va_loss < best_loss:
        best_loss = va_loss; best_state = deepcopy(unet.state_dict())
unet.load_state_dict(best_state)
print(f"Best val loss: {best_loss:.4f}")

---
## Segmentation Metrics: IoU and Dice

In [ ]:
def seg_metrics(pred_logits, true_mask, n_classes=3):
    pred_cls = pred_logits.argmax(1).cpu().numpy()
    true_np  = true_mask.cpu().numpy()
    ious, dices = [], []
    for cls in range(n_classes):
        p = (pred_cls == cls); t = (true_np == cls)
        inter = (p & t).sum(); union = (p | t).sum()
        ious.append(inter / (union + 1e-8))
        dices.append(2*inter / (p.sum() + t.sum() + 1e-8))
    return np.array(ious), np.array(dices)

unet.eval()
all_ious, all_dices = [], []
with torch.no_grad():
    for Xb, yb in val_dl:
        out = unet(Xb.to(device))
        ious, dices = seg_metrics(out, yb)
        all_ious.append(ious); all_dices.append(dices)
mean_iou  = np.mean(all_ious, axis=0)
mean_dice = np.mean(all_dices, axis=0)
print("Segmentation metrics (validation set):")
class_names = ['water','vegetation','bare_soil']
for cls, (iou, dice) in enumerate(zip(mean_iou, mean_dice)):
    print(f"  {class_names[cls]:12s}: IoU={iou:.3f}, Dice={dice:.3f}")
print(f"  {'mIoU':12s}: {mean_iou.mean():.3f}")
print(f"  {'mDice':12s}: {mean_dice.mean():.3f}")

In [ ]:
# Visualise predictions vs ground truth
unet.eval()
sample_X, sample_y = next(iter(val_dl))
with torch.no_grad():
    pred_logits = unet(sample_X.to(device))
    pred_mask   = pred_logits.argmax(1).cpu()
colours = np.array([[0.2,0.4,0.7],[0.2,0.6,0.2],[0.6,0.5,0.3]])
def mask_to_rgb(mask):
    return colours[mask]
fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for col in range(4):
    img_np = sample_X[col].permute(1,2,0).numpy().clip(0,1)
    true_rgb = mask_to_rgb(sample_y[col].numpy())
    pred_rgb = mask_to_rgb(pred_mask[col].numpy())
    iou_col, _ = seg_metrics(pred_logits[col:col+1], sample_y[col:col+1])
    axes[0,col].imshow(img_np); axes[0,col].set_title('Image'); axes[0,col].axis('off')
    axes[1,col].imshow(true_rgb); axes[1,col].set_title('True mask'); axes[1,col].axis('off')
    axes[2,col].imshow(pred_rgb)
    axes[2,col].set_title(f'Pred  mIoU={iou_col.mean():.2f}')
    axes[2,col].axis('off')
plt.suptitle('U-Net: Image / True Mask / Predicted Mask')
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using pixel accuracy as the sole segmentation metric**  
Pixel accuracy is dominated by the most common class. If 80% of pixels are background, a model that predicts background everywhere achieves 80% pixel accuracy. Always report mean IoU (mIoU) or mean Dice across all classes, which weight each class equally regardless of its frequency.

**2. Not concatenating skip connections correctly in U-Net**  
U-Net skip connections concatenate encoder feature maps with upsampled decoder feature maps along the channel dimension. If spatial dimensions do not match exactly (due to odd input sizes), the concatenation fails. Ensure input dimensions are multiples of 2^depth, or use `F.interpolate` to resize before concatenating.

**3. Using CrossEntropyLoss alone for heavily imbalanced segmentation classes**  
If one class covers 5% of pixels (e.g. narrow river channels), CrossEntropyLoss treats all pixels equally and the model ignores the minority class. Combine with Dice loss (which normalises by class frequency) or use class-weighted CrossEntropy to improve minority class performance.

**4. Applying image-level augmentation without identically transforming the mask**  
If the image is randomly flipped or rotated but the corresponding segmentation mask is not identically transformed, the spatial correspondence between image and label is broken. Always apply the same geometric transformation to both image and mask using the same random state.

**5. Upsampling with `nn.Upsample` instead of `nn.ConvTranspose2d` in learned decoders**  
`nn.Upsample` (bilinear or nearest interpolation) is not learnable — it simply resizes without learning how to reconstruct fine spatial details. `ConvTranspose2d` learns the upsampling weights jointly with the rest of the network and generally produces sharper segmentation boundaries.
---
*python_methods_library - Samantha McGarrigle*